# Lab 2.2 — Building a Local Chatbot
**Module II · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_labs/blob/main/module-2-llm/lab2_2_local_chatbot.ipynb)

---

## What you will do
1. Understand how **chat history** works as a list of messages.
2. Implement a `Chatbot` class that maintains conversation context across turns.
3. Use a **system prompt** to give the chatbot a role (a TechRetail customer service agent).
4. See how a system prompt helps with *style* but not with *facts* — motivating RAG.
5. `[Extension]` Handle context window overflow gracefully.

**Estimated time:** 50–65 min

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/DanielFPerez/llm-gnns-course_labs.git",
         "labs_assignment"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "labs_assignment/environment/requirements.txt"], check=True)
    sys.path.insert(0, "labs_assignment")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
from utils.llm import SimpleLLM

llm = SimpleLLM()
print(llm)

---
## 1 · How chat history works

A chat-based LLM does not have persistent memory between calls. The way a chatbot *appears* to remember the conversation is by sending **the entire conversation history** with every new message.

Each message is a dictionary with two keys:
- `"role"`: either `"system"`, `"user"`, or `"assistant"`
- `"content"`: the text of the message

```python
messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user",      "content": "And what is its population?"},
]
```

The LLM sees *all four messages* when answering the last user question, which is why it knows that `"its"` refers to Paris.

In [ ]:
# Provided — run this cell to see multi-turn context in action
messages = [
    {"role": "user", "content": "My name is María."},
    {"role": "assistant", "content": "Nice to meet you, María! How can I help you today?"},
    {"role": "user", "content": "What is my name?"},
]
print(llm.chat(messages, temperature=0))

> **Notice:** The model correctly replies with "María" because it saw the first exchange in the history. Without that history, it would have no idea.

---
## 2 · Building the `Chatbot` class

### Exercise 2.2.1 `[Core]` — Implement `Chatbot`

Implement the `Chatbot` class below. It must:
- Keep an internal list `self.history` of message dicts.
- In `chat(user_message)`: append the user message, call `self.llm.chat(self.history)`, append the assistant reply to history, and return the reply.
- In `reset()`: clear the history (but keep the system prompt if one was set).

The `system_prompt` parameter is optional — if provided, it is inserted as the first message with `role="system"`.

In [ ]:
# YOUR CODE HERE
# Hint: Implement the Chatbot class below. It must:

class Chatbot:
    pass  # replace this

In [ ]:
# Smoke test — make sure multi-turn memory works
bot = Chatbot(llm, temperature=0)
_ = bot.chat("My name is Carlos.")
reply = bot.chat("What did I just tell you?")
print(reply)
print(f"\n(History has {len(bot)} user/assistant exchanges)")

---
## 3 · System prompts: giving the bot a role

A **system prompt** is sent before any user message and tells the model who it is and how to behave. It is the main tool for prompt engineering. Examples:
- *"You are a concise assistant. Answer in at most two sentences."*
- *"You are a Python code reviewer. Always explain your reasoning."*

System prompts affect **style and behaviour**, but they cannot inject facts the model does not already know.

### Exercise 2.2.2 `[Core]` — Customer service bot for TechRetail

Create a `Chatbot` with a system prompt that makes it a TechRetail customer service agent. The prompt should instruct it to:
1. Be polite and professional.
2. Focus only on TechRetail-related questions.
3. When unsure, say so rather than guessing.

Then ask it a general customer service question (e.g., about hours or how to contact support) and see how the persona changes the response style.

In [ ]:
# YOUR CODE HERE
# Hint: Create a Chatbot with a system prompt that makes it a TechRetail customer service agent. The prompt should instruct i...
raise NotImplementedError("Complete this exercise")

In [ ]:
print(customer_bot.chat("Hi, I need help with my order."))

In [ ]:
print(customer_bot.chat("What are your customer support hours?"))

> **Notice the style shift:** The bot is now polite, professional, and on-topic. The system prompt is working. But let's see what happens when we ask something that requires *specific, internal knowledge*.

---
## 4 · The limits of system prompts: facts the model doesn't know

### Exercise 2.2.3 `[Core]` — Ask about specific TechRetail policies

Ask the customer service bot the following questions and compare its answers to the actual TechRetail policies:

| Question | Correct answer |
|---|---|
| What is the return window for a laptop I bought? | **15 days** for electronics |
| How much does same-day delivery cost? | **24,900 COP** |
| What discount do Premium members get? | **10%** on all products |

In [ ]:
# YOUR CODE HERE
# Hint: Ask the customer service bot the following questions and compare its answers to the actual TechRetail policies:
raise NotImplementedError("Complete this exercise")

> **Key insight:** Even with a well-crafted system prompt, the model still gets the specific numbers wrong (or hedges because it correctly doubts itself). The system prompt changed *how* it answers — but it cannot change *what* it knows.
>
> This is the fundamental limitation of prompting alone. We need to *provide* the correct facts — and that is exactly what RAG does.

---
## 5 · Manual grounding: a preview of RAG

### Exercise 2.2.4 `[Core]` — Inject a policy excerpt into the prompt

As a manual experiment that previews what RAG will do automatically, add the relevant policy text directly into the system prompt. Then re-ask the questions.

This shows that the LLM *can* use provided context correctly — the issue is that we can't do this manually at scale.

In [ ]:
# YOUR CODE HERE
# Hint: As a manual experiment that previews what RAG will do automatically, add the relevant policy text directly into the s...
raise NotImplementedError("Complete this exercise")

> **The fix works perfectly** — the model now gives exactly the right numbers because the facts are in its context. The model is not "smarter"; it simply has access to the right information.
>
> **The scalability problem:** TechRetail has 8 policy documents totalling ~1,800 tokens. A real enterprise knowledge base might have thousands of documents and millions of tokens — far too large to paste into every prompt. RAG solves this by *selecting* only the relevant documents at query time.

---
## 6 · `[Extension]` Context window overflow

### Exercise 2.2.5 `[Extension]` — Tracking conversation length

Extend the `Chatbot` class with a `token_count()` method that estimates the total tokens currently in the history. Add a check inside `chat()` that warns the user when the history exceeds a threshold (e.g., 2,000 tokens) — at that point, older messages should be trimmed or the user should start a new session.

Implement a `trim(max_tokens)` method that removes the oldest user/assistant pairs until the total token count is below the threshold (always keeping the system prompt).

In [ ]:
# YOUR CODE HERE
# Hint: Extend the Chatbot class with a token_count() method that estimates the total tokens currently in the history. Add a ...

class ChatbotWithTrimming(Chatbot):
    pass  # replace this

In [ ]:
# Test: have a long enough conversation to trigger trimming
trimming_bot = ChatbotWithTrimming(llm, temperature=0, max_tokens=400)

for i in range(5):
    msg = f"Tell me something interesting about the number {i+1}."
    reply = trimming_bot.chat(msg, max_new_tokens=60)
    print(f"Turn {i+1} | tokens: {trimming_bot.token_count():>4} | messages in history: {len(trimming_bot.history)}")

---
## Summary

| What we built/learned | Why it matters |
|---|---|
| `Chatbot` class with history | Multi-turn memory is just the full message list sent each time |
| System prompt for persona | Controls style and constraints — not factual knowledge |
| Hallucination with persona | Even a role-prompted bot gets specific numbers wrong |
| Manual context injection | Works perfectly, but is not scalable |
| Context window trimming `[Ext]` | Real chatbots must manage history size carefully |

**Next → Lab 2.3:** We build the full RAG pipeline that retrieves the right policy excerpts automatically, so the chatbot always has the correct context — at any scale.